# Mouse-orthologs cross-species concordance

For each mouse pseudotime-APA hit (from `composite_ranking_optimized2_df5.csv` of the mouse pipeline), find the human ortholog and check whether it is also a hit in the Knoblich data.

Pipeline:

1. Load Knoblich composite rankings (both lineages) + mouse composite + ortholog file `mouse_to_human_genes.txt`.
2. Map mouse Ensembl IDs to human Ensembl IDs (one-to-one or one-to-many; keep highest-confidence row per mouse gene).
3. Inner join mouse + human ortholog + Knoblich composite per lineage.
4. Concordance metrics per lineage:
   - 2x2 Fisher exact: (sig in mouse) x (sig in Knoblich), odds ratio with 95% CI.
   - Pearson and Spearman correlation of mouse delta_psi vs Knoblich delta_psi (and rho_W).
   - Direction agreement (% of orthologs with same sign of delta_psi).
5. Plots:
   - Concordance scatter (mouse vs Knoblich) per lineage, points coloured by joint significance.
   - Small-multiples grid for the 5 hand-picked orthologs (HNRNPA2B1, QKI, GPM6B, KLC1, PEA15).
   - Side-by-side 2x2 panels for orthologs significant in both species.
6. Persist `orthologs_significant_both.csv` for use by the ASD notebook (cross-species ASD comparison).

Output: figures under `<utrcalling>/figures/orthologs/` and CSVs under `<utrcalling>/orthologs_results/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 7,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)
print("Setup complete.")

In [ ]:
LINEAGE_TAG = {"Inhibitory neurons": "inhibitory", "Excitatory neurons": "excitatory"}
PADJ_THR = 0.05

KN_BASE = Path(f"{DATA_ROOT}/3utr/organoids/knoblich")
MOUSE_BASE = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses")

KN_COMPOSITE_DIR = KN_BASE / "utrcalling/composite_ranking"
KN_RESULTS_DIR = KN_BASE / "utrcalling/example_genes_results"
KN_FIG_DIR = KN_BASE / "utrcalling/figures/orthologs"
KN_OUT_DIR = KN_BASE / "utrcalling/orthologs_results"
KN_FIG_DIR.mkdir(parents=True, exist_ok=True)
KN_OUT_DIR.mkdir(parents=True, exist_ok=True)

MOUSE_COMPOSITE = MOUSE_BASE / "utrcalling/DirMult_optimized/composite_ranking_optimized2_df5.csv"
ORTHOLOG_FILE = Path(f"{DATA_ROOT}/common_files/annotations/mouse_to_human_genes.txt")

print(f"Knoblich composite:  {KN_COMPOSITE_DIR}")
print(f"Mouse composite:     {MOUSE_COMPOSITE}")
print(f"Ortholog file:       {ORTHOLOG_FILE}")
print(f"Figures out:         {KN_FIG_DIR}")
print(f"Tables out:          {KN_OUT_DIR}")

## Load data


In [ ]:
# Knoblich composite per lineage
kn = {}
for ltag in LINEAGE_TAG.values():
    kn[ltag] = pd.read_csv(KN_COMPOSITE_DIR / f"composite_ranking_optimized2_{ltag}.csv")
    print(f"Knoblich {ltag}: {kn[ltag].shape}")

# Mouse composite. Rename gene_name -> mouse_name immediately so downstream
# tables and figures can refer to mouse_name unambiguously.
mouse = pd.read_csv(MOUSE_COMPOSITE).rename(columns={"gene_name": "mouse_name"})
print(f"Mouse composite: {mouse.shape}")
assert "gene_id" in mouse.columns, "expected gene_id (mouse Ensembl) in mouse composite"
assert "mouse_name" in mouse.columns, "expected mouse_name (renamed from gene_name)"
print(f"Mouse padj < {PADJ_THR}: {(mouse['padj'] < PADJ_THR).sum()}")

# Ortholog file (Ensembl-export CSV, comma-separated, 8 columns)
ortho = pd.read_csv(ORTHOLOG_FILE)
expected_n = 8
assert ortho.shape[1] == expected_n, (
    f"Ortholog file has {ortho.shape[1]} cols, expected {expected_n}. "
    f"Found cols: {list(ortho.columns)}"
)
ortho.columns = [
    "mouse_id",
    "mouse_name",
    "human_id",
    "human_name",
    "homology_type",
    "gene_order_score",
    "wga_coverage",
    "ortho_confidence",
]
print(f"Ortholog file: {ortho.shape}")
print(f"Homology types: {ortho['homology_type'].value_counts(dropna=False).to_dict()}")

## Build mouse -> human mapping


In [ ]:
# Build mouse -> human mapping. Same procedure as in the mouse analysis
# (analysis/mouse_svz/08_260411-optimized2_enrichment_analyses.ipynb): keep one2one or one2many
# orthologs, then sort by (ortho_confidence DESC, wga_coverage DESC) and
# keep one row per mouse Ensembl ID. We do not hard-filter on confidence==1:
# Ensembl assigns 0/1 by combined-score thresholds, but a confidence=0 ortholog
# can still be biologically valid; rather we let the sort prefer confidence=1
# and use lower-confidence rows only when no high-confidence row exists.

o = ortho[ortho["homology_type"].isin(["ortholog_one2one", "ortholog_one2many"])].copy()
print(f"After homology_type filter (one2one+one2many): {len(o):,} rows")
for col in ["ortho_confidence", "wga_coverage"]:
    o[col] = pd.to_numeric(o[col], errors="coerce").fillna(-1.0)
o = o.sort_values(["ortho_confidence", "wga_coverage"], ascending=[False, False])
o_unique = o.drop_duplicates(subset=["mouse_id"], keep="first")
print(f"After dedup by mouse_id: {len(o_unique):,} unique mouse genes mapped")

# Map mouse composite -> human via best-ortholog table.
# Note: o_unique already has mouse_name; mouse already has mouse_name (renamed
# from gene_name above). Drop o_unique's mouse_name to avoid a suffix collision.
mouse_mapped = mouse.merge(
    o_unique[
        [
            "mouse_id",
            "human_id",
            "human_name",
            "homology_type",
            "wga_coverage",
            "ortho_confidence",
        ]
    ],
    left_on="gene_id",
    right_on="mouse_id",
    how="inner",
)
print(f"Mouse genes with at least one human ortholog (in composite): {mouse_mapped.shape[0]:,}")
print(f"  of which sig in mouse (padj<{PADJ_THR}): {(mouse_mapped['padj'] < PADJ_THR).sum():,}")
hc = (mouse_mapped["ortho_confidence"] == 1).sum()
print(f"  ortho_confidence == 1: {hc:,} ({hc / len(mouse_mapped) * 100:.1f}%)")
print(f"  ortho_confidence == 0: {len(mouse_mapped) - hc:,}")
assert "mouse_name" in mouse_mapped.columns, "mouse_name missing after merge"

## Per-lineage join: mouse + human-ortholog + Knoblich


In [ ]:
joined = {}
for ltag in LINEAGE_TAG.values():
    knd = kn[ltag][
        [
            "gene_id",
            "gene_name",
            "padj",
            "delta_psi",
            "rho_W",
            "composite_score_abs",
            "composite_score_signed",
            "trajectory_pattern",
        ]
    ].rename(
        columns={
            "gene_id": "human_id",
            "padj": "padj_kn",
            "delta_psi": "delta_psi_kn",
            "rho_W": "rho_W_kn",
            "composite_score_abs": "comp_abs_kn",
            "composite_score_signed": "comp_signed_kn",
            "trajectory_pattern": "traj_kn",
            "gene_name": "human_name_kn",
        }
    )
    df = mouse_mapped.merge(knd, on="human_id", how="inner", suffixes=("_mouse", "_kn"))
    # Mouse columns to keep
    df = df.rename(
        columns={
            "padj": "padj_mouse",
            "delta_psi": "delta_psi_mouse",
            "rho_W": "rho_W_mouse",
            "composite_score_abs": "comp_abs_mouse",
            "composite_score_signed": "comp_signed_mouse",
        }
    )
    joined[ltag] = df
    print(f"\n{ltag}: {df.shape[0]:,} ortholog rows after join")
    sig_mouse = df["padj_mouse"] < PADJ_THR
    sig_kn = df["padj_kn"] < PADJ_THR
    print(f"  sig in mouse:   {sig_mouse.sum():,}")
    print(f"  sig in Knoblich:{sig_kn.sum():,}")
    print(f"  sig in both:    {(sig_mouse & sig_kn).sum():,}")

## Concordance statistics per lineage


In [ ]:
concordance_rows = []
for ltag, df in joined.items():
    sig_mouse = (df["padj_mouse"] < PADJ_THR).astype(int)
    sig_kn = (df["padj_kn"] < PADJ_THR).astype(int)
    # 2x2 contingency
    a = int(((sig_mouse == 1) & (sig_kn == 1)).sum())
    b = int(((sig_mouse == 1) & (sig_kn == 0)).sum())
    c = int(((sig_mouse == 0) & (sig_kn == 1)).sum())
    d = int(((sig_mouse == 0) & (sig_kn == 0)).sum())
    odds, p_fisher = stats.fisher_exact([[a, b], [c, d]], alternative="greater")
    # Effect-size correlations on the converged subset
    valid = df[["delta_psi_mouse", "delta_psi_kn", "rho_W_mouse", "rho_W_kn"]].dropna()
    pear_dpsi, p_dpsi = (
        stats.pearsonr(valid["delta_psi_mouse"], valid["delta_psi_kn"])
        if len(valid) > 5
        else (np.nan, np.nan)
    )
    spear_dpsi, _ = (
        stats.spearmanr(valid["delta_psi_mouse"], valid["delta_psi_kn"])
        if len(valid) > 5
        else (np.nan, np.nan)
    )
    pear_rho, _ = (
        stats.pearsonr(valid["rho_W_mouse"], valid["rho_W_kn"])
        if len(valid) > 5
        else (np.nan, np.nan)
    )
    spear_rho, _ = (
        stats.spearmanr(valid["rho_W_mouse"], valid["rho_W_kn"])
        if len(valid) > 5
        else (np.nan, np.nan)
    )
    # Direction agreement on sig-in-mouse subset
    sub_dir = df[sig_mouse == 1].dropna(subset=["delta_psi_mouse", "delta_psi_kn"])
    if len(sub_dir):
        dir_agree = float(
            (np.sign(sub_dir["delta_psi_mouse"]) == np.sign(sub_dir["delta_psi_kn"])).mean()
        )
    else:
        dir_agree = np.nan

    concordance_rows.append(
        {
            "lineage": ltag,
            "n_orthologs": len(df),
            "n_sig_mouse": int(sig_mouse.sum()),
            "n_sig_knoblich": int(sig_kn.sum()),
            "n_sig_both": a,
            "fisher_OR": float(odds),
            "fisher_p_one_sided": float(p_fisher),
            "pearson_delta_psi": pear_dpsi,
            "pearson_delta_psi_p": p_dpsi,
            "spearman_delta_psi": spear_dpsi,
            "pearson_rho_W": pear_rho,
            "spearman_rho_W": spear_rho,
            "direction_agreement_in_mouse_sig": dir_agree,
        }
    )
concordance_df = pd.DataFrame(concordance_rows)
concordance_df.to_csv(KN_OUT_DIR / "cross_species_concordance.csv", index=False)
print(concordance_df.to_string(index=False))

## Concordance scatter plots: mouse vs Knoblich effect sizes


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for col_i, (ltag, df) in enumerate(joined.items()):
    sig_mouse = df["padj_mouse"] < PADJ_THR
    sig_kn = df["padj_kn"] < PADJ_THR

    # Top row: delta_psi
    ax = axes[0, col_i]
    cmap = {
        "both": "#B2182B",
        "mouse_only": "#FDB863",
        "knoblich_only": "#2166AC",
        "neither": "lightgrey",
    }
    classes = np.where(
        sig_mouse & sig_kn,
        "both",
        np.where(sig_mouse, "mouse_only", np.where(sig_kn, "knoblich_only", "neither")),
    )
    for cls in ["neither", "mouse_only", "knoblich_only", "both"]:
        m = classes == cls
        ax.scatter(
            df.loc[m, "delta_psi_mouse"],
            df.loc[m, "delta_psi_kn"],
            s=6,
            c=cmap[cls],
            alpha=0.65,
            edgecolors="none",
            rasterized=True,
            label=f"{cls} (n={m.sum()})",
        )
    ax.axhline(0, color="black", lw=0.5, alpha=0.5)
    ax.axvline(0, color="black", lw=0.5, alpha=0.5)
    ax.plot([-1, 1], [-1, 1], color="grey", ls=":", lw=0.6)
    ax.set_xlabel("mouse delta_psi")
    ax.set_ylabel(f"Knoblich {ltag} delta_psi")
    ax.set_title(f"{ltag}: delta_psi concordance")
    ax.legend(fontsize=6, frameon=False, loc="upper left")

    # Bottom row: rho_W
    ax = axes[1, col_i]
    for cls in ["neither", "mouse_only", "knoblich_only", "both"]:
        m = classes == cls
        ax.scatter(
            df.loc[m, "rho_W_mouse"],
            df.loc[m, "rho_W_kn"],
            s=6,
            c=cmap[cls],
            alpha=0.65,
            edgecolors="none",
            rasterized=True,
            label=f"{cls} (n={m.sum()})",
        )
    ax.axhline(0, color="black", lw=0.5, alpha=0.5)
    ax.axvline(0, color="black", lw=0.5, alpha=0.5)
    ax.plot([-1, 1], [-1, 1], color="grey", ls=":", lw=0.6)
    ax.set_xlabel("mouse rho_W")
    ax.set_ylabel(f"Knoblich {ltag} rho_W")
    ax.set_title(f"{ltag}: rho_W concordance")

plt.tight_layout()
fig.savefig(KN_FIG_DIR / "concordance_scatter.pdf", bbox_inches="tight", dpi=300)
fig.savefig(KN_FIG_DIR / "concordance_scatter.png", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
print("saved concordance_scatter.{pdf,png}")

## Save the orthologs-significant-in-both table


In [ ]:
both_rows = []
for ltag, df in joined.items():
    sub = df[(df["padj_mouse"] < PADJ_THR) & (df["padj_kn"] < PADJ_THR)].copy()
    sub["lineage"] = ltag
    both_rows.append(sub)
both_df = pd.concat(both_rows, ignore_index=True) if both_rows else pd.DataFrame()
out_path = KN_OUT_DIR / "orthologs_significant_both.csv"
both_df.to_csv(out_path, index=False)
print(f"Orthologs significant in BOTH species (per lineage): {len(both_df):,}")
print(f"Saved: {out_path}")
if len(both_df):
    cols = [
        "lineage",
        "mouse_id",
        "mouse_name",
        "human_id",
        "human_name_kn",
        "padj_mouse",
        "delta_psi_mouse",
        "rho_W_mouse",
        "padj_kn",
        "delta_psi_kn",
        "rho_W_kn",
        "comp_abs_kn",
    ]
    print("\nTop 20 by Knoblich composite_score_abs:")
    print(both_df.sort_values("comp_abs_kn", ascending=False)[cols].head(20).to_string(index=False))

## Hand-picked orthologs panel: HNRNPA2B1, QKI, GPM6B, KLC1, PEA15


In [ ]:
HAND_PICKED = ["HNRNPA2B1", "QKI", "GPM6B", "KLC1", "PEA15"]

rows = []
for hgene in HAND_PICKED:
    for ltag in LINEAGE_TAG.values():
        df = joined[ltag]
        sub = df[df["human_name_kn"] == hgene]
        if len(sub) == 0:
            continue
        rows.append(
            {
                "human": hgene,
                "lineage": ltag,
                "mouse_name": sub.iloc[0].get("mouse_name", ""),
                "padj_mouse": sub.iloc[0]["padj_mouse"],
                "delta_psi_mouse": sub.iloc[0]["delta_psi_mouse"],
                "padj_kn": sub.iloc[0]["padj_kn"],
                "delta_psi_kn": sub.iloc[0]["delta_psi_kn"],
                "rho_W_kn": sub.iloc[0]["rho_W_kn"],
                "sig_mouse": sub.iloc[0]["padj_mouse"] < PADJ_THR,
                "sig_kn": sub.iloc[0]["padj_kn"] < PADJ_THR,
            }
        )
hand_df = pd.DataFrame(rows)
hand_df.to_csv(KN_OUT_DIR / "hand_picked_orthologs_summary.csv", index=False)
print(hand_df.to_string(index=False))

In [ ]:
# Bar plot: |delta_psi| comparison for hand-picked, per lineage
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, ltag in zip(axes, LINEAGE_TAG.values()):
    sub = hand_df[hand_df["lineage"] == ltag]
    if len(sub) == 0:
        ax.axis("off")
        continue
    x = np.arange(len(sub))
    ax.bar(
        x - 0.2,
        sub["delta_psi_mouse"].abs(),
        width=0.4,
        label="|delta_psi| mouse",
        color="#FDB863",
    )
    ax.bar(
        x + 0.2,
        sub["delta_psi_kn"].abs(),
        width=0.4,
        label="|delta_psi| Knoblich",
        color="#2166AC",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(sub["human"], rotation=30, ha="right")
    ax.set_ylabel("|delta_psi|")
    ax.set_title(f"Hand-picked orthologs: {ltag}")
    ax.legend(fontsize=6, frameon=False)
plt.tight_layout()
fig.savefig(KN_FIG_DIR / "hand_picked_orthologs_dpsi.pdf", bbox_inches="tight", dpi=300)
fig.savefig(KN_FIG_DIR / "hand_picked_orthologs_dpsi.png", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
print("saved hand_picked_orthologs_dpsi.{pdf,png}")

## Summary


In [ ]:
print("Files in figures/orthologs/:")
for f in sorted(KN_FIG_DIR.iterdir()):
    print(f"  {f.name}: {f.stat().st_size / 1024:.0f} KB")
print("\nFiles in orthologs_results/:")
for f in sorted(KN_OUT_DIR.iterdir()):
    print(f"  {f.name}: {f.stat().st_size / 1024:.0f} KB")